# 💬 生成式AI應用開發｜第05週

## 對話狀態、Streaming 與聊天機器人實作

本週延續第 4 週的 prompt 函式，進一步讓模型記得前文、逐步顯示回覆，並完成可重設、可觀察成本的簡易聊天機器人。


> **教師版**：包含完整參考答案、課堂觀察提示與成本防護；建議先讓學生完成練習，再分段揭示答案。


## 🎯 本週學習目標

| # | 完成本週後能做到 | 對應後續課程／應用 |
|---|---|---|
| 1 | 解釋無狀態 API 與對話狀態的差異 | 除錯多輪問答、設計聊天流程 |
| 2 | 使用 `previous_response_id` 串接多輪回覆 | 快速建立原型與客服對話 |
| 3 | 用手動 history 明確管理訊息 | 儲存、裁切、稽核與跨平台整合 |
| 4 | 觀察 context 規模並設計記憶視窗 | 第 6 週成本控制與長對話管理 |
| 5 | 處理 Streaming 事件與文字增量 | 即時 UI、逐字回覆與進度顯示 |
| 6 | 完成具錯誤處理的 CLI 聊天機器人 | 後續 Web／App 聊天介面基礎 |

## 🗺️ 三堂課（150 分鐘）實作流程

| 時間 | 主題 | 實作成果 |
|---:|---|---|
| 0:00–0:20 | 無狀態請求與多輪對話 | 觀察獨立請求為何忘記前文 |
| 0:20–0:50 | `previous_response_id` 與 history | 完成兩種狀態管理方式 |
| 0:50–1:15 | 最小聊天 session | 用少量程式保存上一輪狀態 |
| 1:15–1:45 | 可重設 session 與 usage | 建立較完整的聊天物件 |
| 1:45–2:15 | Streaming events | 累積並即時顯示文字片段 |
| 2:15–2:30 | 必做練習、測試與回顧 | 完成練習 A/B，練習 C 作為進階選做 |

---

## 1️⃣ 環境準備

沿用前兩週設定：API key 放在 Colab Secrets 的 `OPENAI_API_KEY`。本週會產生多次 API 請求，請先確認模型與金鑰，再依序執行示範。


In [ ]:
# 安裝或更新 OpenAI Python SDK
!pip install -q --upgrade openai


In [ ]:
import os
from openai import OpenAI

try:
    from google.colab import userdata
    secret_key = userdata.get("OPENAI_API_KEY")
    if secret_key:
        os.environ["OPENAI_API_KEY"] = secret_key
except ImportError:
    pass
except Exception as exc:
    print(f"無法讀取 Colab Secret：{exc}")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "找不到 OPENAI_API_KEY。請先設定 Colab Secret 或本機環境變數，再重新執行此 cell。"
    )

MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")
client = OpenAI()
print("本週模型：", MODEL)
print("API key 已設定（不顯示內容）")


In [ ]:
CHAT_INSTRUCTIONS = """
# Identity
你是生成式 AI 應用開發課程的助教。
# Instructions
- 使用繁體中文，先直接回答，再補充必要說明。
- 回答控制在 150 字內；不知道時清楚說明限制。
- 不要聲稱記得目前對話以外的資料。
"""


def create_response(user_input, previous_response_id=None, store=True):
    """建立一般 Response；instructions 每一輪都明確重送。"""
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")

    request = {
        "model": MODEL,
        "instructions": CHAT_INSTRUCTIONS,
        "input": user_input,
        "store": store,
    }
    if previous_response_id:
        request["previous_response_id"] = previous_response_id

    try:
        return client.responses.create(**request)
    except Exception as exc:
        raise RuntimeError(f"OpenAI API 呼叫失敗：{exc}") from exc


<div style="border-left:5px solid #f9ab00;padding:10px 14px;background:#fff8e1;">
<font color="#b06000"><b>成本與資料提醒</b></font>：每一輪都是一次 API 請求。即使用 `previous_response_id`，鏈中先前輸入仍會計入 input tokens。Responses 預設可能保存回應以支援串接；課堂資料不得包含個資、成績、密碼或其他敏感資訊。
</div>


---

## 2️⃣ 先看「無狀態」：兩次獨立請求

模型不會因為兩行 Python 程式相鄰，就自動把兩次獨立 API 呼叫視為同一段對話。下面第二次請求沒有附上第一次的內容。


In [ ]:
standalone_1 = create_response("請記住：我叫小安，專題主題是校園活動推薦。")
print("第一輪：", standalone_1.output_text)

standalone_2 = create_response("我叫什麼名字？專題主題是什麼？")
print("第二輪：", standalone_2.output_text)


### ✍️ 觀察紀錄

- 第二輪是否正確說出姓名與主題？
- 如果碰巧答對，答案是否真的來自第一輪，還是模型猜測？
- 應用程式需要保存什麼，才能可靠延續對話？

不要以一次看似正確的輸出判斷「有記憶」；必須檢查下一輪請求實際包含的狀態。


---

## 3️⃣ 三種對話狀態管理方式

| 方式 | 應用程式保存什麼 | 適合情境 | 注意事項 |
|---|---|---|---|
| `previous_response_id` | 上一個 response ID | 同一條短期對話串 | 每輪仍重送 instructions；歷史 token 仍計費 |
| 手動 history | `user`／`assistant` 訊息 | 需要自行裁切、檢查或不保存 Response | 應用程式負責順序與內容 |
| Conversations API | durable conversation ID | 跨工作、裝置或長期 session | 保存期限與隱私策略需另行設計 |

本週實作前兩種；Conversations API 先建立概念，不作為必要練習。


In [ ]:
turn_1 = create_response(
    "請記住：我叫小安，專題主題是校園活動推薦。",
    store=True,
)
print("第一輪：", turn_1.output_text)

turn_2 = create_response(
    "我叫什麼名字？專題主題是什麼？",
    previous_response_id=turn_1.id,
    store=True,
)
print("第二輪：", turn_2.output_text)
print("串接 ID：", turn_1.id, "→", turn_2.id)


### 🔗 `previous_response_id` 的心智模型

```text
使用者第 1 輪 → Response A（id=A）
                         ↓ previous_response_id=A
使用者第 2 輪 → Response B（id=B）
                         ↓ previous_response_id=B
使用者第 3 輪 → Response C
```

`instructions` 只套用在目前這次 response，因此聊天程式應在每一輪明確傳入，不要假設上一輪 instructions 會自動存在。


---

## 4️⃣ 手動管理 conversation history

手動 history 會把需要的 `user` 與 `assistant` 訊息放進下一次 `input`。優點是可檢查、裁切與保存；代價是應用程式必須自行維護順序和內容。

> 進階提醒：若使用需要保留推理項目的模型，正式無狀態流程應依官方文件保存完整 `response.output` 與必要的 encrypted reasoning items。本週先用文字訊息理解基本結構。


In [ ]:
# 方法 B：手動保存訊息 history（結構透明、容易裁切與稽核）
manual_history = [
    {"role": "user", "content": "我叫小華，請記住。"},
]

manual_1 = client.responses.create(
    model=MODEL,
    input=manual_history,
    store=False,
)
print("第一輪：", manual_1.output_text)

manual_history.extend([
    {"role": "assistant", "content": manual_1.output_text},
    {"role": "user", "content": "我叫什麼名字？"},
])

manual_2 = client.responses.create(
    model=MODEL,
    input=manual_history,
    store=False,
)
print("第二輪：", manual_2.output_text)


def history_size(messages):
    """用訊息數與字元數快速觀察 context 規模；實際計費仍以 usage 為準。"""
    chars = sum(len(str(message.get("content", ""))) for message in messages)
    return {"訊息數": len(messages), "總字元數": chars}


print("目前 history 規模：", history_size(manual_history))

### ⚖️ 選擇方式

- 想快速建立同一條短期對話：先用 `previous_response_id`。
- 想自行刪除舊訊息、插入摘要或檢查送出的內容：使用手動 history。
- 想跨 session 保存長期狀態：再評估 Conversations API 或自己的資料庫。

「模型記憶」通常是應用程式把適當資料重新提供給模型，而不是模型永久記住某位使用者。


---

## 5️⃣ 先做一個最小聊天 Session

先用最少的程式保存 `previous_response_id`。這個版本只處理「能不能接續上一輪」，還不處理 transcript、usage、reset 與錯誤保護；下一節會把它擴充成較完整的 class。

In [ ]:
class MinimalChatSession:
    def __init__(self, instructions=CHAT_INSTRUCTIONS):
        self.instructions = instructions
        self.previous_response_id = None

    def ask(self, user_text):
        request = {
            "model": MODEL,
            "instructions": self.instructions,
            "input": user_text,
            "store": True,
        }
        if self.previous_response_id:
            request["previous_response_id"] = self.previous_response_id

        response = client.responses.create(**request)
        self.previous_response_id = response.id
        return response.output_text


minimal_chat = MinimalChatSession()
RUN_MINIMAL_SESSION_DEMO = False
if RUN_MINIMAL_SESSION_DEMO:
    print(minimal_chat.ask("我正在設計課程 FAQ 聊天機器人，請記住。"))
    print(minimal_chat.ask("請重述我正在設計什麼。"))
else:
    print("最小聊天 session 已建立；確認 API 成本後，可將 RUN_MINIMAL_SESSION_DEMO 改為 True。")

---

## 6️⃣ 建立可重設的非串流 Chat Session

延續上一節的最小版本，完整 session 至少要保存：上一個 response ID、畫面要顯示的 transcript、累積 usage，以及 reset 行為。對話失敗時，不應把失敗的回合寫入狀態。

In [ ]:
class ResponseChatSession:
    def __init__(self, instructions=CHAT_INSTRUCTIONS):
        self.instructions = instructions
        self.reset()

    def reset(self):
        self.previous_response_id = None
        self.transcript = []
        self.usage = {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}

    def _add_usage(self, response):
        if response.usage:
            self.usage["input_tokens"] += response.usage.input_tokens
            self.usage["output_tokens"] += response.usage.output_tokens
            self.usage["total_tokens"] += response.usage.total_tokens

    def ask(self, user_text):
        if not isinstance(user_text, str) or not user_text.strip():
            raise ValueError("user_text 必須是非空白字串。")

        request = {
            "model": MODEL,
            "instructions": self.instructions,
            "input": user_text,
            "store": True,
        }
        if self.previous_response_id:
            request["previous_response_id"] = self.previous_response_id

        try:
            response = client.responses.create(**request)
        except Exception as exc:
            raise RuntimeError(f"聊天請求失敗，狀態未更新：{exc}") from exc

        self.previous_response_id = response.id
        self.transcript.extend([
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": response.output_text},
        ])
        self._add_usage(response)
        return response.output_text


chat = ResponseChatSession()
print(chat.ask("我正在做課程 FAQ 聊天機器人。請記住這個主題。"))
print(chat.ask("請用一句話重述我的主題。"))
print("累積 usage：", chat.usage)


### 🧠 簡易記憶不等於無限 history

長對話會遇到三個問題：

1. 舊內容逐漸增加輸入 token 與成本。
2. 過時資訊可能干擾目前任務。
3. 對話可能接近模型 context window。

常見策略包括：只留最近 N 輪、把重要事實另外保存、將舊對話摘要、在長流程使用 compaction。第 5 週先完成最近 N 輪與重要事實的簡化版本。


---

## 7️⃣ Usage、成本與隱私檢核

| 檢查項目 | 課堂做法 |
|---|---|
| 請求次數 | 每執行一個聊天回合就加 1 |
| Token | 記錄 `response.usage`；不要只看輸出長度 |
| 長對話 | 設定回合上限、最近 N 輪或摘要策略 |
| 保存 | 釐清 `store=True/False` 與自己的資料庫保存政策 |
| 隱私 | 不輸入 API key、個資、未公開成績與機密文件 |
| Reset | 清除目前 session 的 ID、transcript 與 usage |

`previous_response_id` 讓程式比較簡潔，但不代表先前 token 免費，也不取代應用程式的隱私與保存政策。


---

## 8️⃣ Streaming：先看到片段，再取得完整結果

非串流會等待完整回答後一次回傳；Streaming 會以 SSE 事件逐步送回資料。文字介面最常處理：

- `response.created`：串流開始。
- `response.output_text.delta`：新增文字片段，會出現多次。
- `response.completed`：回應完成，可取得 response ID 與 usage。
- `error`：串流失敗，不應更新成功狀態。


In [ ]:
def stream_response(
    user_input,
    previous_response_id=None,
    instructions=CHAT_INSTRUCTIONS,
    on_delta=None,
):
    """即時處理文字 delta，回傳（完整文字, completed response）。"""
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("user_input 必須是非空白字串。")

    request = {
        "model": MODEL,
        "instructions": instructions,
        "input": user_input,
        "store": True,
        "stream": True,
    }
    if previous_response_id:
        request["previous_response_id"] = previous_response_id

    chunks = []
    completed_response = None
    callback = on_delta or (lambda delta: print(delta, end="", flush=True))

    try:
        for event in client.responses.create(**request):
            if event.type == "response.output_text.delta":
                chunks.append(event.delta)
                callback(event.delta)
            elif event.type == "response.completed":
                completed_response = event.response
            elif event.type == "error":
                raise RuntimeError(f"串流事件錯誤：{event}")
    except Exception as exc:
        raise RuntimeError(f"Streaming 失敗：{exc}") from exc

    if completed_response is None:
        raise RuntimeError("串流結束但沒有收到 response.completed。")
    return "".join(chunks), completed_response


### 🧩 為什麼要同時「顯示」與「累積」？

畫面即時顯示只解決 UX；應用程式仍需要完整文字，才能寫入 transcript、保存紀錄、進行後續檢查或交給下一個步驟。因此每個 `delta` 都要顯示並加入 `chunks`，最後再 `"".join(chunks)`。


In [ ]:
streamed_text, streamed_response = stream_response(
    "請用三個短句說明 Streaming 對聊天機器人的好處。"
)
print("\n\n--- 串流完成 ---")
print("完整字數：", len(streamed_text))
print("Response ID：", streamed_response.id)
if streamed_response.usage:
    print("Usage：", streamed_response.usage)


In [ ]:
class StreamingChatSession(ResponseChatSession):
    def ask_stream(self, user_text, on_delta=None):
        full_text, response = stream_response(
            user_text,
            previous_response_id=self.previous_response_id,
            instructions=self.instructions,
            on_delta=on_delta,
        )
        self.previous_response_id = response.id
        self.transcript.extend([
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": full_text},
        ])
        self._add_usage(response)
        return full_text


def run_console_chat(session):
    """輸入 /reset 清除狀態，輸入 /exit 結束；Notebook 中請手動取消過久輸入。"""
    print("聊天開始：輸入 /reset 重設，輸入 /exit 結束。")
    while True:
        user_text = input("\n你：").strip()
        if user_text == "/exit":
            print("聊天結束。")
            break
        if user_text == "/reset":
            session.reset()
            print("已清除目前對話狀態。")
            continue
        if not user_text:
            print("請輸入內容。")
            continue

        print("AI：", end="", flush=True)
        try:
            session.ask_stream(user_text)
            print()
        except RuntimeError as exc:
            print(f"\n錯誤：{exc}")


streaming_chat = StreamingChatSession()
# 取消下一行註解即可啟動互動聊天；結束時輸入 /exit。
# run_console_chat(streaming_chat)


### 🛑 中斷與錯誤處理

- 串流可能在完成前斷線；不要把半段回答當成成功結果。
- 只有收到 `response.completed` 後，才更新 `previous_response_id` 或對話 history。
- 捕捉 `response.failed`、`error` 與網路例外，向使用者顯示可重試訊息。
- Streaming 的部分輸出比完整回答更難做內容審核；正式應用需設計輸入／輸出審核與中止策略。

---

## 9️⃣ 課堂練習 A（必做）：課程諮詢 Session

建立 `CourseAdvisorSession`：固定課程助教角色、支援兩輪追問、可查看 transcript 與 usage，並能用 `reset()` 開始新對話。這是本週第一個必做練習。

In [ ]:
COURSE_ADVISOR_INSTRUCTIONS = """
# Identity
你是生成式 AI 應用開發課程的諮詢助教。
# Instructions
- 使用繁體中文回答課程學習問題。
- 先確認學生目前做到哪一步，再提供最多三個可執行步驟。
- 不知道課程規定時，請學生詢問授課教師，不要自行編造。
"""


class CourseAdvisorSession(ResponseChatSession):
    def __init__(self):
        super().__init__(instructions=COURSE_ADVISOR_INSTRUCTIONS)


advisor = CourseAdvisorSession()
RUN_PRACTICE_A = False
if RUN_PRACTICE_A:
    print(advisor.ask("我會 Python 函式，但還不熟 API。下一步該做什麼？"))
    print(advisor.ask("請把剛才的第一步拆成兩個小步驟。"))
    print("Transcript：", advisor.transcript)
    print("Usage：", advisor.usage)


---

## 🔟 課堂練習 B（必做）：串流回覆與完整文字

建立 `ask_with_chunk_count()`：逐段顯示回覆、計算收到幾個 delta，最後回傳完整文字與 response。錯誤或未完成時不可假裝成功。這是本週第二個必做練習。

In [ ]:
def ask_with_chunk_count(user_text, previous_response_id=None):
    chunk_count = 0

    def show_and_count(delta):
        nonlocal chunk_count
        chunk_count += 1
        print(delta, end="", flush=True)

    full_text, response = stream_response(
        user_text,
        previous_response_id=previous_response_id,
        on_delta=show_and_count,
    )
    print(f"\n\n共收到 {chunk_count} 個文字 delta。")
    return full_text, response


RUN_PRACTICE_B = False
if RUN_PRACTICE_B:
    practice_text, practice_response = ask_with_chunk_count(
        "請用 120 字說明多輪對話與 Streaming 的差異。"
    )
    print("完整字數：", len(practice_text))


---

## 1️⃣1️⃣ 課堂練習 C（選做）：最近 N 輪＋重要事實

進階選做：使用手動 history 建立 `WindowedMemoryChat`。重要事實放在 instructions；一般對話只保留最近 `max_turns` 輪，避免 history 無限制成長。若課堂時間不足，可作為課後挑戰。

In [ ]:
class WindowedMemoryChat:
    def __init__(self, important_facts=None, max_turns=3):
        if max_turns <= 0:
            raise ValueError("max_turns 必須是正整數。")
        self.important_facts = list(important_facts or [])
        self.max_turns = max_turns
        self.turns = []

    def build_instructions(self):
        facts = "\n".join(f"- {fact}" for fact in self.important_facts) or "- 無"
        return CHAT_INSTRUCTIONS + f"\n# Important facts\n{facts}"

    def build_input(self, user_text):
        recent_turns = self.turns[-self.max_turns:]
        messages = []
        for turn in recent_turns:
            messages.extend([
                {"role": "user", "content": turn["user"]},
                {"role": "assistant", "content": turn["assistant"]},
            ])
        messages.append({"role": "user", "content": user_text})
        return messages

    def ask(self, user_text):
        if not isinstance(user_text, str) or not user_text.strip():
            raise ValueError("user_text 必須是非空白字串。")
        response = client.responses.create(
            model=MODEL,
            instructions=self.build_instructions(),
            input=self.build_input(user_text),
            store=False,
        )
        self.turns.append({"user": user_text, "assistant": response.output_text})
        return response.output_text

    def reset_recent_turns(self):
        self.turns.clear()


memory_chat = WindowedMemoryChat(
    important_facts=["使用者偏好繁體中文", "專題主題是校園活動推薦"],
    max_turns=3,
)
RUN_PRACTICE_C = False
if RUN_PRACTICE_C:
    print(memory_chat.ask("我的專題主題是什麼？"))


### 🔍 Context 管理檢查

先用 `history_size()` 觀察訊息數與總字元數，再搭配回應的 `usage` 判讀實際 token 使用量。接著把 `max_turns` 設成 `1` 執行遺忘實驗：若較早提供的名字已被裁切，模型就不應再可靠地回答。字元數只是方便比較，不等同 token 數或費用。

---

## 1️⃣2️⃣ 聊天機器人測試計畫

至少測試：第一輪、依賴前文的追問、reset 後不可記得舊資料、空白輸入、API 錯誤、串流中斷、長對話與敏感資料提醒。

| 測試 | 預期行為 |
|---|---|
| 先說姓名，再問姓名 | 同一 session 可回答 |
| reset 後再問姓名 | 不應沿用舊 session |
| 傳入空白 | 程式先拒絕，不呼叫 API |
| 串流正常完成 | delta 合併後等於完整顯示內容 |
| 串流失敗 | 不更新 previous response ID |
| 超過保留回合數 | 只送出最近 N 輪與重要事實 |

In [ ]:
chatbot_test_cases = [
    "請用一句話介紹生成式 AI。",
    "請列出剛才回答中的一個關鍵詞。",
]


def run_chatbot_smoke_test():
    """付費 smoke test：確認串流完成後才更新狀態。"""
    bot = StreamingChatSession()
    for prompt in chatbot_test_cases:
        print(f"\n使用者：{prompt}")
        bot.ask(prompt)
        assert bot.previous_response_id is not None
    print("\n✅ Smoke test 完成")


def run_forgetting_experiment():
    """四次付費請求：示範 max_turns=1 如何裁切較早的對話。"""
    forgetful = WindowedMemoryChat(max_turns=1)
    for message in ["我叫小華。", "我喜歡吃拉麵。", "今天天氣很好。"]:
        print("使用者：", message)
        print("AI：", forgetful.ask(message))

    probe_input = forgetful.build_input("我叫什麼名字？")
    print("送入模型的 history 規模：", history_size(probe_input))
    print("AI：", forgetful.ask("我叫什麼名字？"))


RUN_FORGETTING_EXPERIMENT = False
if RUN_FORGETTING_EXPERIMENT:
    run_forgetting_experiment()
else:
    print("遺忘實驗尚未執行；確認 4 次 API 成本並完成 WindowedMemoryChat 後，再改為 True。")


RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    run_chatbot_smoke_test()
else:
    print("Smoke test 尚未執行；確認 API 成本並完成類別後，再改為 True。")

## ✅ 完成檢核

- [ ] 能說明無狀態 API 為何不會自動記得上一輪。
- [ ] 能用 `previous_response_id` 或手動 history 完成多輪對話。
- [ ] 能解釋為何 `instructions` 需要在每次請求重新提供。
- [ ] 能辨識 `response.output_text.delta` 與 `response.completed`。
- [ ] 只在串流成功完成後更新對話狀態。
- [ ] 用 `history_size()` 與 `max_turns=1` 實驗觀察 context 裁切。
- [ ] 聊天機器人能處理 `exit`、例外與重設狀態。

## 📝 課後任務：完成個人聊天機器人

選擇「課程 FAQ、學習助教、專題規劃或校園資訊」之一，繳交：

1. 角色與 instructions。
2. 至少三輪、其中一輪依賴前文的對話紀錄。
3. Streaming 顯示與完整文字累積。
4. reset、空白輸入與 API 錯誤處理。
5. 至少五筆測試案例與 usage 紀錄。
6. 150 字說明你的記憶、成本與隱私策略。

<div style="border-left:5px solid #d93025;padding:10px 14px;background:#fce8e6;">
<font color="#b3261e"><b>安全提醒</b></font>：不得提交 API key、真實個資、未公開成績、醫療或財務敏感資料。測試請使用虛構資訊。
</div>


## 📚 延伸閱讀

- OpenAI Conversation state：https://developers.openai.com/api/docs/guides/conversation-state
- OpenAI Streaming responses：https://developers.openai.com/api/docs/guides/streaming-responses
- OpenAI Responses API reference：https://developers.openai.com/api/docs/api-reference/responses

下週將進入：**Structured Outputs、JSON Schema 與資料抽取**。
